# Analyse NLP des Rapports de Collecte
Objectif : Prédire la catégorie du déchet en se basant uniquement sur les descriptions textuelles fournies dans `Rapport_Collecte`.


In [28]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

## Chargement des données textuelles


In [29]:
train = pd.read_csv('../data/processed/train.csv')
val = pd.read_csv('../data/processed/val.csv')

# Nettoyage basique : suppression des NaN dans le texte
train['Rapport_Collecte'] = train['Rapport_Collecte'].fillna('N/A')
val['Rapport_Collecte'] = val['Rapport_Collecte'].fillna('N/A')

X_train_text = train['Rapport_Collecte']
y_train = train['Categorie']
X_val_text = val['Rapport_Collecte']
y_val = val['Categorie']

In [30]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from wordcloud import WordCloud
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
french_stopwords = stopwords.words('french')
stemmer = SnowballStemmer('french')

def preprocess_text(text):
    tokens = word_tokenize(str(text).lower(), language='french')
    tokens = [stemmer.stem(t) for t in tokens if t.isalpha() and t not in french_stopwords]
    return ' '.join(tokens)

train['text_clean'] = train['Rapport_Collecte'].apply(preprocess_text)
val['text_clean'] = val['Rapport_Collecte'].apply(preprocess_text)
display(train[['Rapport_Collecte', 'text_clean']].head(3))

,Rapport_Collecte,text_clean
0,Déchet en verre collecté en provenance de l'Us...,déchet verr collect proven poid lourd kg volum...
1,Ferraille ou métal collecté issu de la collect...,ferraill métal collect issu collect citoyen ma...
2,Déchet métallique détecté issu de la collecte ...,déchet métall détect issu collect citoyen poid...


In [31]:
from wordcloud import WordCloud
import plotly.express as px
from PIL import Image
import numpy as np

text_all = ' '.join(train['text_clean'])
wc = WordCloud(width=800, height=400, background_color='white').generate(text_all)
wc_array = wc.to_array()
fig_wc = px.imshow(wc_array, title='WordCloud des Rapports de Collecte')
fig_wc.update_layout(coloraxis_showscale=False)
fig_wc.update_xaxes(showticklabels=False)
fig_wc.update_yaxes(showticklabels=False)
fig_wc.show()

In [32]:
from gensim.models import Word2Vec
sentences = [text.split() for text in train['text_clean']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, seed=42)
def get_w2v_vector(text):
    words = text.split()
    vectors = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)
X_train_w2v = np.array([get_w2v_vector(t) for t in train['text_clean']])
X_val_w2v = np.array([get_w2v_vector(t) for t in val['text_clean']])

In [33]:
from gensim.models import FastText
ft_model = FastText(sentences, vector_size=100, window=5, min_count=1, seed=42)
def get_ft_vector(text):
    words = text.split()
    vectors = [ft_model.wv[w] for w in words if w in ft_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)
# Using get_ft_vector correctly for FastText, user accidentally wrote get_w2v_vector in prompt but this fixes it
X_train_ft = np.array([get_ft_vector(t) for t in train['text_clean']].copy())
X_val_ft = np.array([get_ft_vector(t) for t in val['text_clean']].copy())

## Comparaison Vectoriseurs x Classifieurs
Nous testons systématiquement le Bag of Words (BoW) et le TF-IDF avec différents algorithmes pour trouver la meilleure combinaison.


In [34]:
from sklearn.preprocessing import MinMaxScaler
vectorizers = {
    'BoW': CountVectorizer(max_features=5000, stop_words=french_stopwords),
    'TF-IDF': TfidfVectorizer(max_features=5000, stop_words=french_stopwords)
}

classifiers = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Linear SVC': LinearSVC()
}

nlp_results = []

# Traditional Vectorizers
for vec_name, vec in vectorizers.items():
    X_train_vec = vec.fit_transform(train['text_clean'])
    X_val_vec = vec.transform(val['text_clean'])
    
    for clf_name, clf in classifiers.items():
        clf.fit(X_train_vec, y_train)
        preds = clf.predict(X_val_vec)
        
        acc = accuracy_score(y_val, preds)
        f1 = f1_score(y_val, preds, average='weighted')
        
        nlp_results.append({
            'Vectorizer': vec_name,
            'Classifier': clf_name,
            'Accuracy': acc,
            'F1-Score': f1,
            'Model': clf,
            'Vec': vec
        })
        print(f"{vec_name} + {clf_name:<20} -> Acc: {acc:.4f}")

# Embedding Vectorizers
custom_vecs = {
    'Word2Vec': (X_train_w2v, X_val_w2v),
    'FastText': (X_train_ft, X_val_ft)
}

scaler = MinMaxScaler()

for vec_name, (X_train_vec, X_val_vec) in custom_vecs.items():
    for clf_name, clf in classifiers.items():
        # Naive Bayes requires non-negative inputs, embeddings can be negative
        if clf_name == 'Naive Bayes':
            X_tr = scaler.fit_transform(X_train_vec)
            X_va = scaler.transform(X_val_vec)
        else:
            X_tr = X_train_vec
            X_va = X_val_vec
            
        clf.fit(X_tr, y_train)
        preds = clf.predict(X_va)
        
        acc = accuracy_score(y_val, preds)
        f1 = f1_score(y_val, preds, average='weighted')
        
        nlp_results.append({
            'Vectorizer': vec_name,
            'Classifier': clf_name,
            'Accuracy': acc,
            'F1-Score': f1,
            'Model': clf,
            'Vec': vec_name
        })
        print(f"{vec_name} + {clf_name:<20} -> Acc: {acc:.4f}")

BoW + Naive Bayes          -> Acc: 1.0000
BoW + Logistic Regression  -> Acc: 1.0000
BoW + Linear SVC           -> Acc: 1.0000
TF-IDF + Naive Bayes          -> Acc: 1.0000
TF-IDF + Logistic Regression  -> Acc: 1.0000
TF-IDF + Linear SVC           -> Acc: 1.0000
Word2Vec + Naive Bayes          -> Acc: 0.8558
Word2Vec + Logistic Regression  -> Acc: 1.0000
Word2Vec + Linear SVC           -> Acc: 1.0000
FastText + Naive Bayes          -> Acc: 0.8284
FastText + Logistic Regression  -> Acc: 1.0000
FastText + Linear SVC           -> Acc: 1.0000


## Synthèse des résultats NLP


In [35]:
df_nlp = pd.DataFrame(nlp_results)
fig_nlp = px.bar(
    df_nlp, 
    x='Classifier', y='Accuracy', color='Vectorizer', 
    barmode='group', title='Performance des combinaisons NLP',
    height=500
)
fig_nlp.show()

display(df_nlp[['Vectorizer', 'Classifier', 'Accuracy', 'F1-Score']].sort_values('Accuracy', ascending=False).round(4))

,Vectorizer,Classifier,Accuracy,F1-Score
0,BoW,Naive Bayes,1.0000,1.0000
1,BoW,Logistic Regression,1.0000,1.0000
2,BoW,Linear SVC,1.0000,1.0000
3,TF-IDF,Naive Bayes,1.0000,1.0000
4,TF-IDF,Logistic Regression,1.0000,1.0000
5,TF-IDF,Linear SVC,1.0000,1.0000
7,Word2Vec,Logistic Regression,1.0000,1.0000
8,Word2Vec,Linear SVC,1.0000,1.0000
10,FastText,Logistic Regression,1.0000,1.0000
11,FastText,Linear SVC,1.0000,1.0000


## Sauvegarde du meilleur modèle NLP


In [36]:
best_nlp_idx = df_nlp['Accuracy'].idxmax()
best_nlp_model = df_nlp.loc[best_nlp_idx, 'Model']
best_vec = df_nlp.loc[best_nlp_idx, 'Vec']

print(f"Meilleure combinaison : {df_nlp.loc[best_nlp_idx, 'Vectorizer']} + {df_nlp.loc[best_nlp_idx, 'Classifier']}")

os.makedirs('../models', exist_ok=True)
joblib.dump(best_nlp_model, '../models/best_nlp_model.joblib')
joblib.dump(best_vec, '../models/nlp_vectorizer.joblib')
print("Modèle et Vectoriseur sauvegardés dans ../models/")

Meilleure combinaison : BoW + Naive Bayes
Modèle et Vectoriseur sauvegardés dans ../models/


## Conclusion
L'utilisation de techniques NLP sur les rapports de collecte permet d'extraire une information précieuse qui complète les mesures physiques. Une approche hybride (fusionnant Features Physiques + Textuelles) pourrait encore améliorer la précision globale du système.
